# Exploración de transacciones

Exploración de `transactions` (y un vistazo a `call_transcripts`, `service_agents` y `enum_mappings`) sobre el warehouse local de silver (`data/warehouse.duckdb`, ver `make silver`).

- Carga las 4,4 M de transacciones en pandas: requiere varios GB de memoria.
- Consultas de referencia: `query_silver.ipynb`. Problemas conocidos de los datos: `docs/known_issues.md`.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

# repo root: the first folder upwards that holds data/ (works from the root or from pipeline/notebooks)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
con = duckdb.connect(str(ROOT / "data/warehouse.duckdb"), read_only=True)

In [2]:
df_tables = con.sql("""
    select table_name, estimated_size as filas, column_count as columnas
    from duckdb_tables()
    order by estimated_size desc
""").df()
df_tables

,table_name,filas,columnas
0,transactions,4425008,28
1,campaign_sends,1746801,28
2,call_center_interactions,686296,27
3,products,400000,21
4,satisfaction_surveys,212759,26
5,call_transcripts,171321,24
6,customers,150000,31
7,complaints,67095,33
8,daily_exchange_rates,13164,11
9,service_agents,1200,22


In [3]:

df = con.sql("""
    select *
    from call_transcripts
    limit 10000
""").df()

In [4]:
tables = list(df_tables.table_name)

In [5]:
df_tables

,table_name,filas,columnas
0,transactions,4425008,28
1,campaign_sends,1746801,28
2,call_center_interactions,686296,27
3,products,400000,21
4,satisfaction_surveys,212759,26
5,call_transcripts,171321,24
6,customers,150000,31
7,complaints,67095,33
8,daily_exchange_rates,13164,11
9,service_agents,1200,22


In [6]:
df_transactions = con.sql("""
    select *
    from transactions

""").df()

In [7]:
# transactions tiene un producto, el cliente, lar

In [8]:
df_transactions.iloc[0]

transaction_id                                   TRX-0012RNBNIDX7W1SSRWRX
transaction_date                                      2025-08-19 03:00:23
process_date                                          2025-08-18 00:00:00
product_id                                               PRD-F0JFI5UGYHS2
customer_id                                              CLI-0GQ945NIY4IU
transaction_type                                                 Transfer
transaction_category                                                  NaN
amount                                                            5609.49
currency                                                              USD
amount_usd                                                            NaN
channel                                                               POS
branch_id                                                             NaN
merchant_name                                                         NaN
merchant_category                     

In [9]:
df_transactions.currency.value_counts(dropna=False)

currency
USD    2437979
COP    1194444
ARS     792585
Name: count, dtype: int64

In [10]:
df_transactions.groupby(["transaction_type","transaction_category","merchant_category"],dropna=False).size()

transaction_type  transaction_category  merchant_category
Adjustment        NaN                   NaN                  132118
Deposit           NaN                   NaN                  609409
Payment           Entertainment         NaN                  105580
                  Food                  NaN                  175555
                  Health                NaN                   70313
                  Other                 NaN                  105356
                  Services              NaN                  140213
                  Transport             NaN                  105218
                  NaN                   NaN                   36729
Purchase          Entertainment         Entertainment        146307
                                        NaN                    7756
                  Food                  Food                 243898
                                        NaN                   13015
                  Health                Health            

In [11]:
df_transactions.transaction_type.value_counts(dropna=False)

transaction_type
Purchase      1083406
Withdrawal     964673
Transfer       896438
Payment        738964
Deposit        609409
Adjustment     132118
Name: count, dtype: int64

In [12]:
df_transactions.transaction_category.value_counts(dropna=False)

transaction_category
NaN              2693520
Food              432468
Services          345370
Other             260518
Transport         260316
Entertainment     259643
Health            173173
Name: count, dtype: int64

In [13]:
df_transactions.groupby(["transaction_type","transaction_category"],dropna=False).size()

transaction_type  transaction_category
Adjustment        NaN                     132118
Deposit           NaN                     609409
Payment           Entertainment           105580
                  Food                    175555
                  Health                   70313
                  Other                   105356
                  Services                140213
                  Transport               105218
                  NaN                      36729
Purchase          Entertainment           154063
                  Food                    256913
                  Health                  102860
                  Other                   155162
                  Services                205157
                  Transport               155098
                  NaN                      54153
Transfer          NaN                     896438
Withdrawal        NaN                     964673
dtype: int64

In [14]:
df_transactions["dif_date"] = df_transactions.transaction_date-df_transactions.process_date

In [15]:
df_transactions["dif_date"].describe()

count                   4425008
mean     0 days 18:00:08.770814
std      0 days 06:55:35.223673
min             0 days 06:00:00
25%             0 days 12:00:25
50%             0 days 18:00:11
75%             1 days 00:00:01
max             1 days 06:00:00
Name: dif_date, dtype: object

In [16]:
df_transactions.groupby([ "year","month"],dropna=False).size()

year  month
2023  6         57418
      7        121121
      8        127919
      9        118902
      10       128065
      11       122255
      12       125490
2024  1        122418
      2        117330
      3        121105
      4        121731
      5        126416
      6        121441
      7        129316
      8        122109
      9        118012
      10       127758
      11       120835
      12       123343
2025  1        122056
      2        110632
      3        123365
      4        120838
      5        126679
      6        118998
      7        126867
      8        125886
      9        118785
      10       129256
      11       120323
      12       122817
2026  1        121549
      2        115332
      3        124296
      4        126499
      5        127155
      6         70691
dtype: int64

In [17]:
df_transactions.customer_id.nunique()

134515

In [18]:
df_transactions.product_id.nunique()

339963

In [19]:
len(df_transactions)/(df_transactions.customer_id.nunique())

32.896019031334795

In [20]:
df_transactions["branch_id_null"] =df_transactions["branch_id"].isna()

In [21]:
df_transactions.groupby(["channel","branch_id_null"],dropna=False).size()

channel   branch_id_null
ATM       False             1262007
          True                66327
App       True               663414
Branch    False              125925
          True                 6570
POS       True              1548161
Transfer  True                89159
Web       True               663445
dtype: int64

In [22]:
df_transactions.transaction_status.value_counts(dropna=False)

transaction_status
Approved    4070681
Declined     221234
Pending       88343
Reversed      44750
Name: count, dtype: int64

In [23]:
df_transactions.groupby([ "transaction_type","transaction_status"],dropna=False).size()

transaction_type  transaction_status
Adjustment        Approved              121394
                  Declined                6688
                  Pending                 2708
                  Reversed                1328
Deposit           Approved              560724
                  Declined               30240
                  Pending                12268
                  Reversed                6177
Payment           Approved              679954
                  Declined               36851
                  Pending                14684
                  Reversed                7475
Purchase          Approved              996168
                  Declined               54565
                  Pending                21595
                  Reversed               11078
Transfer          Approved              824676
                  Declined               44979
                  Pending                17841
                  Reversed                8942
Withdrawal        Appro

In [24]:
df_transactions.is_fraud.value_counts()

is_fraud
False    4420692
True        4316
Name: count, dtype: int64

In [25]:
df_transactions.groupby([ "transaction_type","is_fraud"],dropna=False).size()

transaction_type  is_fraud
Adjustment        False        131974
                  True            144
Deposit           False        608802
                  True            607
Payment           False        738234
                  True            730
Purchase          False       1082320
                  True           1086
Transfer          False        895640
                  True            798
Withdrawal        False        963722
                  True            951
dtype: int64

In [26]:
df_transactions.iloc[0]

transaction_id                                   TRX-0012RNBNIDX7W1SSRWRX
transaction_date                                      2025-08-19 03:00:23
process_date                                          2025-08-18 00:00:00
product_id                                               PRD-F0JFI5UGYHS2
customer_id                                              CLI-0GQ945NIY4IU
transaction_type                                                 Transfer
transaction_category                                                  NaN
amount                                                            5609.49
currency                                                              USD
amount_usd                                                            NaN
channel                                                               POS
branch_id                                                             NaN
merchant_name                                                         NaN
merchant_category                     

In [27]:
import pandas as pd
df_transactions.transaction_date[0].month

8

In [28]:
#year y año vienen de process_date

In [29]:
df_transactions.isna().sum()/len(df_transactions)*100

transaction_id            0.000000
transaction_date          0.000000
process_date              0.000000
product_id                0.000000
customer_id               0.000000
transaction_type          0.000000
transaction_category     60.870398
amount                    0.000000
currency                  0.000000
amount_usd               57.343535
channel                   0.000000
branch_id                68.634362
merchant_name            76.740517
merchant_category        76.750483
transaction_country       0.000000
transaction_city         10.002490
transaction_status        0.000000
response_code             4.995087
is_fraud                  0.000000
fraud_score              20.003512
latitude                 80.625391
longitude                80.626182
year                      0.000000
month                     0.000000
_dq_issues                0.000000
_source_key               0.000000
_ingested_at            100.000000
_run_id                 100.000000
dif_date            

In [30]:

df_service_agents = con.sql("""
    select *
    from service_agents

""").df()

In [31]:
df_service_agents.iloc[0]

agent_id                                           AGT-04JD8RS9U4
employee_code                                              E12152
first_name                                         Mercedes Julia
last_name                                         Suárez González
email                                    msuarez@bancohotmail.com
phone                                            +57 300 162 3701
native_accent                                           colombian
country_of_origin                                        Colombia
assigned_branch_id                                   SUC-3YPRGN58
agent_type                                                  Phone
experience_level                                           Senior
languages                                                 español
specialty                                                Créditos
hire_date                                     2023-02-15 00:00:00
avg_csat                                                     4.34
total_mont

In [32]:

df_enum = con.sql("""
    select *
    from enum_mappings
""").df()

In [33]:
df_enum 

,table_name,column_name,raw_value,canonical_value
0,customers,document_type,Pasaporte,Passport
1,customers,country,México,Mexico
2,branches,country,México,Mexico
3,transactions,transaction_country,México,Mexico
4,campaign_sends,open_country,México,Mexico
5,branches,geographic_zone,Urbana,Urban
6,branches,geographic_zone,Suburbana,Suburban
7,products,product_type,Cuenta Corriente,Checking Account
8,products,product_type,Cuenta Ahorro,Savings Account
9,products,product_type,Tarjeta Crédito,Credit Card
